In [2]:
### Exported fine-tuned model saved in hugging face format
import json, torch
from pathlib import Path
from transformers import VisionEncoderDecoderModel, TrOCRProcessor


In [3]:
print("CWD:", Path.cwd())

CWD: /home/jovyan/DL/12433693_text-recognition-german/dicts


In [4]:
model_name = "microsoft/trocr-base-printed"

ckpt_path = Path("../src/best_model.pth")
out_dir = Path("../src/artifacts/trocr_finetuned")


In [5]:
processor = TrOCRProcessor.from_pretrained(model_name, use_fast=True)
model = VisionEncoderDecoderModel.from_pretrained(model_name)

Some weights of VisionEncoderDecoderModel were not initialized from the model checkpoint at microsoft/trocr-base-printed and are newly initialized: ['encoder.pooler.dense.bias', 'encoder.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [6]:
# Load finetuned checkpoint dict
ckpt = torch.load(ckpt_path, map_location="cpu")
model.load_state_dict(ckpt["model_state_dict"])

# Apply config tweaks (same as training notebook)
model.config.decoder_start_token_id = processor.tokenizer.cls_token_id
model.config.pad_token_id = processor.tokenizer.pad_token_id
model.config.vocab_size = model.config.decoder.vocab_size

In [7]:
out_dir.mkdir(parents=True, exist_ok=True)
model.save_pretrained(out_dir)
processor.save_pretrained(out_dir)

[]

In [8]:
# Save generation settings for the Streamlit app to reuse
gen_cfg = {
  "max_length": 64,
  "num_beams": 4,
  "early_stopping": False,
  "temperature": 0.7,
  "do_sample": True,
  "top_k": 50,
  "top_p": 0.95
}

In [9]:
with open(out_dir / "generation_config.json", "w", encoding="utf-8") as f:
    json.dump(gen_cfg, f, indent=2)

print("Saved offline model+processor to:", out_dir.resolve())
print("Checkpoint used:", ckpt_path.resolve())
print("Artifact sample files:", sorted([p.name for p in out_dir.iterdir()])[:15])


Saved offline model+processor to: /home/jovyan/DL/12433693_text-recognition-german/src/artifacts/trocr_finetuned
Checkpoint used: /home/jovyan/DL/12433693_text-recognition-german/src/best_model.pth
Artifact sample files: ['config.json', 'generation_config.json', 'merges.txt', 'model.safetensors', 'preprocessor_config.json', 'special_tokens_map.json', 'tokenizer.json', 'tokenizer_config.json', 'vocab.json']
